# The Negotiation: AI Agent Deal-Making

In this exercise, you'll write a negotiation strategy for an AI agent that will negotiate a **$50M corporate loan** on your behalf. Your agent (the Borrower) will face off against a Bank agent in a multi-round negotiation.

**You only need to edit ONE cell** — the strategy prompt. Everything else runs automatically.

---

In [ ]:
# =============================================================
#  SETUP — Run this cell first, then listen to the instructor!
# =============================================================

!pip install transformers accelerate bitsandbytes torch -q

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Check GPU
if not torch.cuda.is_available():
    print("WARNING: No GPU detected. Falling back to smaller model.")
    MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
    if gpu_mem < 14:
        print("Limited GPU memory. Falling back to smaller model.")
        MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading {MODEL_NAME} with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"\nModel loaded successfully! You're ready to negotiate.")

In [ ]:
# @title Helper Functions (do not edit)

import json
import re
import time
from IPython.display import display, HTML, clear_output


def generate_response(system_prompt, conversation_history):
    """Generate a response given a system prompt and conversation history."""
    messages = [{"role": "system", "content": system_prompt}] + conversation_history
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response


def extract_deal_terms(transcript):
    """Use the LLM to extract final deal terms from the negotiation transcript."""
    extraction_prompt = (
        "You are a legal analyst. Read the negotiation transcript below and extract "
        "the FINAL agreed-upon terms. If the negotiation failed or no deal was reached, "
        "respond with exactly: NO_DEAL\n\n"
        "Otherwise respond with ONLY a JSON object (no other text) with these keys:\n"
        '- "interest_rate": number (percentage, e.g. 6.5)\n'
        '- "loan_term": number (years, e.g. 7)\n'
        '- "collateral": one of "full", "partial", "minimal", "none"\n'
        '- "covenants": one of "strict", "moderate", "light", "none"\n'
        '- "prepayment_penalty": number (percentage, e.g. 1.5)\n'
        '- "origination_fee": number (percentage, e.g. 0.75)\n\n'
        "TRANSCRIPT:\n" + transcript
    )
    messages = [{"role": "user", "content": extraction_prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if "NO_DEAL" in raw.upper():
        return None

    # Try to parse JSON from the response
    try:
        # Find JSON object in response
        json_match = re.search(r'\{[^{}]*\}', raw, re.DOTALL)
        if json_match:
            terms = json.loads(json_match.group())
        else:
            terms = json.loads(raw)

        # Validate and sanitize
        defaults = {
            "interest_rate": 7.8,
            "loan_term": 5,
            "collateral": "full",
            "covenants": "strict",
            "prepayment_penalty": 3.0,
            "origination_fee": 1.25,
        }
        for key, default in defaults.items():
            if key not in terms:
                terms[key] = default

        # Clamp numeric values to reasonable ranges
        terms["interest_rate"] = max(3.0, min(12.0, float(terms["interest_rate"])))
        terms["loan_term"] = max(1, min(15, int(terms["loan_term"])))
        terms["prepayment_penalty"] = max(0, min(5.0, float(terms["prepayment_penalty"])))
        terms["origination_fee"] = max(0, min(3.0, float(terms["origination_fee"])))

        # Normalize categorical values
        terms["collateral"] = terms["collateral"].lower().strip()
        if terms["collateral"] not in ("full", "partial", "minimal", "none"):
            terms["collateral"] = "full"
        terms["covenants"] = terms["covenants"].lower().strip()
        if terms["covenants"] not in ("strict", "moderate", "light", "none"):
            terms["covenants"] = "strict"

        return terms
    except (json.JSONDecodeError, ValueError, KeyError):
        return None


def score_deal(terms):
    """Deterministic scoring of deal terms (0-100, higher = better for borrower)."""
    if terms is None:
        return {"total": 0, "status": "FAILED", "details": {}}

    # Interest rate: offered 7.8%, best possible ~5.0%
    rate_score = max(0, min(100, (7.8 - terms["interest_rate"]) / (7.8 - 5.0) * 100))

    # Loan term: offered 5yr, best possible 10yr
    term_score = max(0, min(100, (terms["loan_term"] - 3) / (10 - 3) * 100))

    # Collateral
    collateral_map = {"full": 0, "partial": 50, "minimal": 80, "none": 100}
    collateral_score = collateral_map.get(terms["collateral"], 0)

    # Covenants
    covenant_map = {"strict": 0, "moderate": 50, "light": 80, "none": 100}
    covenant_score = covenant_map.get(terms["covenants"], 0)

    # Prepayment penalty: offered 3%, lower is better
    prepay_score = max(0, min(100, (3.0 - terms["prepayment_penalty"]) / 3.0 * 100))

    # Origination fee: offered 1.25%, lower is better
    orig_score = max(0, min(100, (1.25 - terms["origination_fee"]) / 1.25 * 100))

    # Weighted total
    total = (
        rate_score * 0.30
        + term_score * 0.20
        + collateral_score * 0.15
        + covenant_score * 0.15
        + prepay_score * 0.10
        + orig_score * 0.10
    )

    return {
        "total": round(total, 1),
        "status": "AGREED",
        "details": {
            "Interest Rate": {"score": round(rate_score, 1), "value": f"{terms['interest_rate']:.2f}%"},
            "Loan Term": {"score": round(term_score, 1), "value": f"{terms['loan_term']} yrs"},
            "Collateral": {"score": collateral_score, "value": terms["collateral"].title()},
            "Covenants": {"score": covenant_score, "value": terms["covenants"].title()},
            "Prepayment Penalty": {"score": round(prepay_score, 1), "value": f"{terms['prepayment_penalty']:.1f}%"},
            "Origination Fee": {"score": round(orig_score, 1), "value": f"{terms['origination_fee']:.2f}%"},
        },
    }


def get_rating(score):
    """Return a fun rating based on the total score."""
    if score >= 90:
        return "Wall Street Legend"
    elif score >= 80:
        return "Master Negotiator"
    elif score >= 70:
        return "Solid Dealmaker"
    elif score >= 60:
        return "Getting There"
    elif score >= 50:
        return "The Bank Won This One"
    elif score >= 40:
        return "Back to Business School"
    else:
        return "You Got Eaten Alive"


def generate_highlight(transcript):
    """Generate a highlight reel comment about the negotiation."""
    prompt = (
        "You are a witty sports commentator narrating a business negotiation. "
        "Read the transcript below and write 2-3 sentences highlighting the most "
        "interesting, dramatic, or funny moment. Be entertaining.\n\n"
        "TRANSCRIPT:\n" + transcript
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print("Helper functions loaded.")

---

# THE $50M LOAN NEGOTIATION

You are **TechForward Inc.**, a mid-size fintech company seeking a **$50M term loan** to fund your expansion into European markets.

## Your Situation
- **Annual revenue:** $120M, growing 25% YoY
- **Current debt:** $15M (small revolving credit facility)
- **Credit rating:** BBB+
- You have offers from **two other banks** (your BATNA), but this bank offers the best relationship and potential for future business
- You need the money within **60 days**

## The Bank
- **MegaBank Corp**, a top-10 commercial bank
- They want your business (you're a hot fintech) but have strict risk policies
- Their typical terms for your profile: 6.5-8.5% interest, 5-7 year term
- You don't know their exact flexibility (but it exists!)

## Negotiable Terms
| Term | Currently Offered |
|------|------------------|
| Interest rate | 7.8% |
| Loan term | 5 years |
| Collateral | Full asset pledge |
| Financial covenants | Strict EBITDA and leverage ratios |
| Prepayment penalty | 3% (years 1-2), 1.5% (years 3-5) |
| Origination fee | 1.25% |

## Your Goal
**Get the best deal possible for TechForward Inc.** Write a strategy for your AI negotiation agent below, then run the negotiation and see how you score!

---

In [ ]:
# @title Bank Agent Prompt (do not edit)

bank_strategy = """
You are a Senior Relationship Manager at MegaBank Corp, a top-10 commercial bank.
You are negotiating a $50M term loan with TechForward Inc., a growing fintech company.

YOUR ROLE:
You represent the bank. Your goal is to close the deal at terms favorable to MegaBank
while keeping the client happy enough to maintain a long-term relationship.

CURRENT OFFER ON THE TABLE:
- Interest rate: 7.8%
- Loan term: 5 years
- Collateral: Full asset pledge
- Covenants: Strict EBITDA (>3.0x) and leverage (<2.5x) ratios, quarterly reporting
- Prepayment penalty: 3% in years 1-2, 1.5% in years 3-5
- Origination fee: 1.25%

YOUR HIDDEN FLEXIBILITY (do NOT reveal these limits, negotiate gradually toward them):
- Interest rate: Can go as low as 5.8% but only if borrower offers something in return
- Loan term: Can extend to 10 years maximum
- Collateral: Can reduce to partial asset pledge if other terms are strong
- Covenants: Can relax to moderate (EBITDA >2.5x, leverage <3.0x) but not eliminate
- Prepayment penalty: Can reduce to 1.0% flat, can waive entirely for strong packages
- Origination fee: Can reduce to 0.50% minimum

NEGOTIATION STRATEGY:
1. ANCHORING: Start with the current offer. Present it as already competitive.
   Say things like "We've already sharpened our pencils on this one."
2. CONCESSIONS: Make concessions slowly, one at a time. Always ask for something
   in return. "I can look at the rate, but I'd need you to consider a full asset pledge."
3. WALK-AWAY: If the borrower is extremely unreasonable (demanding below 4% interest,
   or refusing all covenants AND all collateral, or being rude/threatening), politely
   end the negotiation. Say something like: "I don't think we can find common ground
   today. Perhaps one of your other banks can accommodate those terms."
4. PERSONALITY: Be professional but human. Occasionally say things like:
   - "Look, I went to bat for you with the credit committee on this."
   - "Between you and me, this is a strong offer."
   - "I want to make this work. Help me help you."
5. REACT TO STYLE:
   - If borrower is AGGRESSIVE or THREATENING: Get more rigid. Make fewer concessions.
     Say "I understand you have options, but strong-arming isn't going to get us to yes."
   - If borrower is COLLABORATIVE: Be more open. Offer creative solutions like
     step-down rates or performance-based covenant relief.
   - If borrower BLUFFS POORLY (e.g., claims they have 3% offers from other banks
     when that's unrealistic for BBB+ credit): Call it gently. "I'd be surprised if
     anyone is offering sub-4% for this credit profile in the current market."
   - If borrower gives GOOD BUSINESS REASONING: Be persuaded. "That's a fair point
     about your growth trajectory. Let me see what I can do."
6. PACING: In early rounds, hold firm. In middle rounds, start showing flexibility.
   In later rounds, push toward closing. "We've been going back and forth — let me
   put my best and final offer on the table."

IMPORTANT RULES:
- Keep responses to 3-5 sentences. Be concise.
- Always address specific terms the borrower raises.
- When making a concession, state the new number clearly.
- If you reach agreement, clearly state "I think we have a deal" and list all final terms.
- Never break character. You ARE the banker.
"""

print("Bank agent loaded.")

In [ ]:
# ================================================================
#  YOUR NEGOTIATION STRATEGY
#  Edit the text between the triple quotes below.
#  This is the ONLY cell you need to change!
# ================================================================

borrower_strategy = """
You are the CFO of TechForward Inc. negotiating a $50M loan with MegaBank Corp.

YOUR STRATEGY:
[Write your negotiation strategy here. Think about:]
- What's your opening position? (aim high!)
- What tactics will you use? (be specific)
- What's your walk-away point?
- What creative solutions can you propose?
- How will you build rapport with the banker?

EXAMPLE (delete this and write your own):
Start by expressing enthusiasm about the partnership. Open by asking for
5.5% interest rate and a 10-year term. Emphasize our growth trajectory
and the competitive offers from other banks. Be willing to accept
slightly stricter covenants in exchange for a lower rate.
"""

print("Borrower strategy set! Now run the next cell to start the negotiation.")

In [ ]:
# ================================================================
#  RUN NEGOTIATION
# ================================================================

MAX_ROUNDS = 8

# Conversation histories from each agent's perspective
bank_history = []      # Bank sees borrower as "user", itself as "assistant"
borrower_history = []  # Borrower sees bank as "user", itself as "assistant"
transcript_parts = []

flavor_texts = [
    "The banker adjusts their cufflinks...",
    "A pause as both sides consider their positions...",
    "The banker glances at their notes...",
    "You can hear the clock ticking on the conference room wall...",
    "The banker leans back in their chair...",
    "A brief, tense silence fills the room...",
    "The banker takes a sip of water...",
    "Both sides exchange a knowing look...",
]

deal_reached = False
walk_away = False

print("=" * 60)
print("  THE NEGOTIATION BEGINS")
print("=" * 60)
print()

try:
    for round_num in range(1, MAX_ROUNDS + 1):
        print(f"--- ROUND {round_num} of {MAX_ROUNDS} ---")
        print()

        # --- Bank speaks ---
        bank_msg = generate_response(bank_strategy, bank_history)
        print(f"\033[94mBANK:\033[0m {bank_msg}")
        print()

        # Update histories
        bank_history.append({"role": "assistant", "content": bank_msg})
        borrower_history.append({"role": "user", "content": bank_msg})
        transcript_parts.append(f"BANK: {bank_msg}")

        # Check for walk-away or deal
        bank_lower = bank_msg.lower()
        if any(phrase in bank_lower for phrase in [
            "can't find common ground", "cannot find common ground",
            "end this negotiation", "not going to work",
            "walk away", "perhaps one of your other banks",
            "unable to proceed"
        ]):
            print("\033[91mThe bank has walked away from the negotiation!\033[0m")
            walk_away = True
            break
        if "we have a deal" in bank_lower or "deal is done" in bank_lower:
            deal_reached = True
            break

        # Dramatic pause
        time.sleep(1)
        print(f"  \033[90m{flavor_texts[(round_num - 1) % len(flavor_texts)]}\033[0m")
        print()

        # --- Borrower speaks ---
        borrower_msg = generate_response(borrower_strategy, borrower_history)
        print(f"\033[92mBORROWER:\033[0m {borrower_msg}")
        print()

        # Update histories
        borrower_history.append({"role": "assistant", "content": borrower_msg})
        bank_history.append({"role": "user", "content": borrower_msg})
        transcript_parts.append(f"BORROWER: {borrower_msg}")

        # Check for walk-away or deal
        borrower_lower = borrower_msg.lower()
        if any(phrase in borrower_lower for phrase in [
            "walk away", "no deal", "we're done here",
            "going with another bank", "ending this negotiation"
        ]):
            print("\033[91mYou walked away from the negotiation!\033[0m")
            walk_away = True
            break
        if "we have a deal" in borrower_lower or "deal is done" in borrower_lower:
            deal_reached = True
            break

        time.sleep(1)

    if not deal_reached and not walk_away:
        print("\033[93mTime's up! The negotiation has reached the maximum number of rounds.\033[0m")
        print("Extracting whatever terms were last discussed...")

    print()
    print("=" * 60)
    print("  NEGOTIATION COMPLETE")
    print("=" * 60)

    full_transcript = "\n\n".join(transcript_parts)

except Exception as e:
    print(f"\n\033[91mThe negotiation broke down due to a communication failure. Try again!\033[0m")
    print(f"Error details: {e}")
    full_transcript = "\n\n".join(transcript_parts) if transcript_parts else "No transcript."

In [ ]:
# ================================================================
#  SCORING
# ================================================================

if walk_away or not transcript_parts:
    deal_terms = None
else:
    print("Analyzing the deal terms...")
    deal_terms = extract_deal_terms(full_transcript)

result = score_deal(deal_terms)

print()
print("=" * 60)
print("  NEGOTIATION RESULTS")
print("=" * 60)
print()

if result["status"] == "FAILED":
    print(f"Deal Status:    \033[91mFAILED\033[0m")
    print()
    print(f"TOTAL SCORE:    \033[91m0/100\033[0m")
    print()
    print(f'RATING: "{get_rating(0)}"')
else:
    print(f"Deal Status:    \033[92mAGREED\033[0m")
    print()
    print(f"{'Term':<22} {'Offered':<12} {'Final':<12} {'Score':<8}")
    print("-" * 56)

    offered = {
        "Interest Rate": "7.80%",
        "Loan Term": "5 yrs",
        "Collateral": "Full",
        "Covenants": "Strict",
        "Prepayment Penalty": "3.0%",
        "Origination Fee": "1.25%",
    }

    for term_name, info in result["details"].items():
        print(f"{term_name:<22} {offered[term_name]:<12} {info['value']:<12} {info['score']:.0f}/100")

    print("-" * 56)
    total = result["total"]
    color = "\033[92m" if total >= 70 else "\033[93m" if total >= 50 else "\033[91m"
    print(f"{'TOTAL SCORE:':<46} {color}{total:.0f}/100\033[0m")
    print()
    print(f'RATING: "{get_rating(total)}"')

# Generate highlight reel
print()
print("-" * 60)
if transcript_parts:
    print("HIGHLIGHT REEL:")
    highlight = generate_highlight(full_transcript)
    print(f"\033[3m{highlight}\033[0m")
print("-" * 60)

---

# Prompt Engineering Tips for Negotiation

Want to improve your score? Try these strategies in your prompt:

1. **BE SPECIFIC:** "Ask for 5.5% interest" beats "ask for a lower rate"
2. **SET ANCHORS:** Start with an ambitious opening position
3. **USE BATNA:** Mention your alternatives ("We have offers from other banks")
4. **TRADE, DON'T JUST ASK:** "I'll accept stricter covenants for a lower rate"
5. **BUILD RAPPORT:** "Express excitement about the long-term relationship"
6. **BE CREATIVE:** Propose structures the bank hasn't considered
7. **READ THE ROOM:** Tell your agent to adjust if the banker seems resistant
8. **DON'T BLUFF BADLY:** If you claim something unrealistic, the bank may call it

Go back to your strategy cell, edit it, and run the negotiation again!

---